In [0]:
print("helo")

Auto Loader in Databricks

Auto Loader is Databricks’ cloud-native file ingestion engine for ingesting new files incrementally from object storage.

Supported Sources:

AWS S3
Azure ADLS Gen2
Google Cloud Storage (GCS)

Modes:

Directory listing - Directory listing scans storage paths to detect new files (This works in free edition)
File notification - Processes files as soon as they arrive at scale (This will not work in free edition because the cloud storage event trigger can't control/trigger Databricks LF Ingestion) (LIKE FILE WATCHER)
Directory listing (Databricks Lakeflow Ingestion - Autoloader - Directory Listing)

Spark lists the Cloud directory (pull model)
Detects new files (Incremental Autoloader)
Infers schema / evolves if needed
Copy the file(s) & store the schema info in a schema file, so further schema inference is not needed.
After file1 is copied to Bronze layer -> Updates checkpoint (maintaining the file info of whichever is copied already)
Waits for next trigger of the Lakeflow pipeline and follow step 1 to 5.
File Notification (we will see it in the cloud databricks version)

Cloud storage emits file-create event (S3 Event, ADLS Event Grid, GCS Pub/Sub)
Event is delivered to Databricks queue
Auto Loader receives notification (push model)
New file is registered
Infers schema / evolves if needed
Copy the file(s) & store the schema info in a schema file, so further schema inference is not needed.
Updates checkpoint (file1 is processed...)
Stream stays idle until next event arrives
Benifits of Autoloader:

Incremental and Efficient File Ingestion: Auto Loader automatically detects and processes new files as they arrive in your source directory (e.g., S3 or Unity Catalog volume). This eliminates manual tracking and reprocessing, ensuring only new data is ingested each run.

Schema Evolution Support: With options like "cloudFiles.schemaEvolutionMode": "addNewColumns" and "mergeSchema": "true", Auto Loader can handle changes in your data schema over time, adding new columns without breaking your pipeline.

Scalability and Resource Optimization: Properties such as "cloudFiles.maxFilesPerTrigger" allow you to control how many files are processed per batch, helping manage resource usage and scale to large datasets.

Checkpointing and Fault Tolerance: Auto Loader maintains checkpoints and schema locations, so it can resume from where it left off in case of failures, ensuring reliable and consistent data ingestion.

Unified Streaming and Batch Processing: By using readStream and writeStream, your pipeline can handle both streaming and batch workloads seamlessly, making it suitable for real-time and scheduled data ingestion.

To perform schema evolution, we have to use the below properties:
Read side:
.option("cloudFiles.schemaEvolutionMode","addNewColumns")
Write side:
.option("mergeSchema", "true")

In [0]:
#We learn Autoloading of Incremental data from cloud source, Schema evolution, 
cloudsrc="/Volumes/lakehousecat1/deltadb/wd37src_datalake/landing"#s3 storage path
bronzetgt="/Volumes/lakehousecat1/deltadb/wd37src_datalake/bronze"
#To resolve it, you must use a data source that is accessible from your AWS-based Databricks workspace, such as an S3 bucket or a Unity Catalog volume.
ckptlocation="/Volumes/lakehousecat1/deltadb/wd37src_datalake/_checkpoint"#stores the files copied information post write is successful
schemalocation="/Volumes/lakehousecat1/deltadb/wd37src_datalake/_schema"#stores the inferred schema of the source data

# # using dbutils create all directory
# dbutils.fs.rm(cloudsrc,True)
# dbutils.fs.rm(bronzetgt,True)
# dbutils.fs.rm(ckptlocation,True)
# dbutils.fs.rm(schemalocation,True)


dbutils.fs.mkdirs(cloudsrc)
dbutils.fs.mkdirs(bronzetgt)
dbutils.fs.mkdirs(ckptlocation)
dbutils.fs.mkdirs(schemalocation)

In [0]:
# spark.read.option(k,v).format("csv").load() - spark sql , batch mode 

# spark.readStream.option(k,v).format("csv").load()  - streaming mode , structure streaming -micro

# spark.readStream.format("cloudFiles")  - streaming mode , auto loader - microbatch
# streaming source , auto loader is provoiding 

# spark.read.format("parquet")  - batch mode 

# spark.readStream.format("parquet") - streaming mode 

# cloudFiles - streaming file source , supports csv, json, avro,etc ..
# cloudFiles.format - specify file type (csv , json , parquet...)
# cloudFiles.maxFilesPerTrigger - per fetch how many files it will bring from source 
# cloudFiles.inferColumnTypes -True( inferschema ), False (default all columns are string type )
# cloudFiles.schemaLocation  - loc , schemma will be stored in this location , track the schema changes 
# checkpointLocation - checkpoint location for the stream , maintains the state of the stream 
# cloudFiles.schemaEvolutionMode - addNewColumns(default) , fail , rescue (coulnnameforcoruptedrecord=_rescued_data), none
# permissive + coulnnameforcoruptedrecord , dropmalfoormed , failfast 


df1=spark.readStream.format("cloudFiles")\
.option("cloudFiles.format","csv")\
.option("cloudFiles.maxFilesPerTrigger",1)\
.option("cloudFiles.inferColumnTypes",True)\
.option("cloudFiles.schemaEvolutionMode","failOnNewColumns")\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("header",True)\
.load(cloudsrc)#this can be s3/adls/gcs
#.option("cloudFiles.useNotifications", "true") (Remove this option to enable directory listing)
#maxFilesPerTrigger - this property help spark to process howmany files in an iteration to control the resource utilization (all files will be processed ultimately)




In [0]:

#realtime trigger is not possible in free serverless
#writeStream will read data from df1 (materialized here) and write to bronzetgt using the schema generated by reader and checkpoint info stored
# default trigger -RT, immediate 
# processingTime- time interval - 1 min 
# once - deprcated 
# availableNow = True  -> current state load , similar to once  , like a batch 

# checkpiontLocation - checkpoint location for the stream , maintains the state of the stream 


#df1.writeStream.trigger(availableNow=True)\
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.start(bronzetgt)


#option("mergeSchema", "true") \

# write into table     
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.table("lakehousecat1.deltadb.tblemp_auto")

In [0]:

%sql
select * from lakehousecat1.deltadb.tblemp_auto

In [0]:
spark.read.format("delta").load(bronzetgt).show(100)

In [0]:
%sql
select * from cloud_files_state("/Volumes/lakehousecat1/deltadb/datalake/wd37src/_checkpoint")

In [0]:

%fs head /Volumes/lakehousecat1/deltadb/datalake/wd37src/_checkpoint/offsets/0

In [0]:

%fs head /Volumes/lakehousecat1/deltadb/datalake/wd37src/_schema/_schemas/0

In [0]:
with open(cloudsrc+"/emp_14_2.csv","w") as f:
    f.write("id,name,age,city_name\n")
    f.write("14,emp14,24,New York\n")
    f.write("15,emp15,25,New York\n")
    f.write("16,emp16,26,New York\n")
    f.close()



with open(cloudsrc+"/emp_14_7.csv","w") as f:
    f.write("id,name,age,city_name,dept,year,month\n")
    f.write("abc,emp14,24,New York,1,2026,6\n")
    f.close()

In [0]:
with open(cloudsrc+"/emp_dtype_error2.csv","w") as f:
    f.write("id,name,age\n")
    f.write("100,rishi,24\n")
    f.write("100a,babu,24\n")
    f.close()


with open(cloudsrc+"/emp_newcol2.csv","w") as f:
    f.write("id,name,age,dept,city\n")
    f.write("200,rishi,24,1,chen\n")
    f.write("201,babu,24,2,chen\n")
    f.close()